In [ ]:
import csv
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def scrape_ikman_cars():
    # Setup Chrome options (Headless mode so no window pops up)
    chrome_options = Options()
    # chrome_options.add_argument("--headless") 

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    # The URL from your screenshot (Cars in Sri Lanka)
    url = "https://ikman.lk/en/ads/sri-lanka/cars"
    driver.get(url)
    
    # Wait for the page to load content
    time.sleep(5) 

    # Find the container for ad items
    # ikman usually uses a specific class for the list items
    ads = driver.find_elements(By.CSS_SELECTOR, "li[class*='list-item']")[:10]
    
    data_list = []
    
    for ad in ads:
        try:
            # Extracting Title, Price, and Details
            title = ad.find_element(By.CSS_SELECTOR, "h2").text
            price = ad.find_element(By.CSS_SELECTOR, "div[class*='price']").text
            
            # The info string usually contains "Mileage, Year, Location"
            details = ad.find_element(By.CSS_SELECTOR, "div[class*='description']").text
            
            data_list.append({
                "Title": title,
                "Price": price,
                "Details": details.replace("\n", " | ")
            })
        except Exception:
            continue # Skip if an element isn't found (like an inline ad)

    # Save to CSV
    if data_list:
        with open('ikman_check.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=["Title", "Price", "Details"])
            writer.writeheader()
            writer.writerows(data_list)
        print(f"Successfully saved {len(data_list)} items to ikman_check.csv")
    else:
        print("No data found. The site may have updated its layout.")

    driver.quit()

if __name__ == "__main__":
    scrape_ikman_cars()